# PDelta2-ER Layer Lab — close the remaining Transformer gap

This notebook starts from the strongest previous direction: **PDelta2 F96 + causal Conv4**. It tests exactly three targeted improvements rather than adding unrelated complexity.

1. **Learnable per-feature retention** — feature channels are initialized with log-spaced half-lives from 8 to 2048 tokens and remain trainable.
2. **Residual error memory** — a small 16/24-feature recurrent state is trained on `Transformer attention - detached main PDelta output`. Its output gain starts at exactly zero.
3. **Teacher-error-directed training** — the hardest 25% of tokens by current teacher divergence are upweighted, while LM refinement gives stronger weight to true next-token CE.

The combined quality candidate is **`retention_residual24_f96_hard`**. The experiment still selects only from validation and opens held-out test documents afterwards. A strict win requires the complete paired 95% bootstrap interval of candidate-minus-Transformer NLL to be below zero.


In [ ]:
import importlib, pathlib, subprocess, sys, tempfile

REF = 'main'
WORK = pathlib.Path('/content') if pathlib.Path('/content').exists() else pathlib.Path.cwd()
REPO = pathlib.Path(tempfile.mkdtemp(prefix='TinyCeNN-er-lab-', dir=WORK))
subprocess.run(['git','clone','--depth','1','--branch',REF,'https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'transformers==4.57.6','datasets>=3,<5','pandas','matplotlib','pytest>=8'], check=True)
for p in (REPO, REPO/'src'):
    if str(p) not in sys.path: sys.path.insert(0, str(p))
importlib.invalidate_caches()
print('Commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'], text=True).strip())


## 1 · Choose the experimental budget

Use **balanced** first. `quick` checks the complete pipeline cheaply. `strong` is intended only after balanced identifies a promising ingredient.


In [ ]:
from datetime import datetime, timezone
import torch

PROFILE = 'balanced'   # quick | balanced | strong
LAYER = 18
TRAIN_CONTEXT = 256
TEST_CONTEXTS = '256,512,1024,2048'
SEED = 2026

assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> GPU'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT_DIR = WORK / 'TinyCeNN-er-results' / f'{PROFILE}-{RUN_ID}'
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0))
print('Output:', OUTPUT_DIR)


## 2 · Numerical, causal, streaming and checkpoint checks

These tests verify the retention spectrum, hard-token weighting, zero-init residual branch, correct Conv4 streaming tail, FP16 persistent memory / FP32 curvature, checkpoint reconstruction and direct CLI launch.


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q',str(REPO/'tests/test_pdelta2_er.py')], cwd=REPO, check=True)


## 3 · Train and evaluate the ingredient ablation

Balanced mode compares the previous Conv4 baseline, teacher-error training alone, retention alone, residual memory alone, and two combined PDelta2-ER variants. Child-process logs are streamed so failures show their real traceback instead of only `CalledProcessError`.


In [ ]:
cmd = [
    sys.executable, str(REPO/'scripts/benchmark_pdelta2_er_layer.py'),
    '--profile', PROFILE, '--layer', str(LAYER),
    '--train-context', str(TRAIN_CONTEXT), '--test-contexts', TEST_CONTEXTS,
    '--seed', str(SEED), '--output-dir', str(OUTPUT_DIR),
]
print(' '.join(cmd), flush=True)
process = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
returncode = process.wait()
if returncode:
    raise RuntimeError(f'Benchmark exited with status {returncode}')


## 4 · Inspect which ingredient moved the Pareto frontier


In [ ]:
import json, pandas as pd
from IPython.display import display

selection = json.loads((OUTPUT_DIR/'selection.json').read_text())
validation = pd.read_csv(OUTPUT_DIR/'validation_summary.csv')
tests = pd.read_csv(OUTPUT_DIR/'test_summary.csv')
effects = pd.read_csv(OUTPUT_DIR/'ingredient_effects.csv')
retention = pd.read_csv(OUTPUT_DIR/'retention_summary.csv')
focus = pd.read_csv(OUTPUT_DIR/'teacher_error_focus.csv')
print('Selection:', json.dumps(selection, indent=2))
print('\nValidation ranking')
display(validation[['name','ingredient','validation_delta_nll','validation_ppl','output_cosine','state_vs_transformer_fp16_256','decode_step_ms']])
print('\nHeld-out tests')
display(tests[['name','context','delta_nll','ci95_low','ci95_high','ppl_ratio','state_vs_transformer_fp16','verdict']])
print('\nIngredient improvement relative to previous Conv4 baseline')
display(effects.sort_values('delta_nll_improvement_vs_conv4_baseline', ascending=False))
print('\nLearned retention half-lives')
display(retention)
print('\nTeacher-error / residual diagnostics')
display(focus)


In [ ]:
import matplotlib.pyplot as plt

ranked = validation.sort_values('validation_delta_nll')
plt.figure(figsize=(11,5))
plt.bar(ranked['name'], ranked['validation_delta_nll'])
plt.axhline(0, linewidth=1)
plt.ylabel('Validation delta NLL vs Transformer')
plt.xticks(rotation=35, ha='right')
plt.title('Can any PDelta2-ER candidate cross below Transformer?')
plt.tight_layout()
plt.show()

long = tests[tests['name'] != 'transformer_exact_trainable_control'].copy()
plt.figure(figsize=(9,5))
for name, frame in long.groupby('name'):
    if len(frame) > 1:
        frame = frame.sort_values('context')
        plt.plot(frame['context'], frame['ppl_ratio'], marker='o', label=name)
plt.axhline(1.0, linewidth=1)
plt.xlabel('Context')
plt.ylabel('PPL ratio candidate / Transformer')
plt.title('Long-context quality')
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 5 · Package and download every result

The ZIP includes reports, CSVs, selection metadata and checkpoints. The browser download starts automatically when the run is complete.


In [ ]:
import shutil

ZIP_BASE = WORK / f'pdelta2-er-{OUTPUT_DIR.name}'
ZIP_PATH = pathlib.Path(shutil.make_archive(str(ZIP_BASE), 'zip', root_dir=OUTPUT_DIR))
print('ZIP:', ZIP_PATH, 'size MB:', round(ZIP_PATH.stat().st_size / 1e6, 2))
try:
    from google.colab import files
    files.download(str(ZIP_PATH))
except Exception as exc:
    print('Automatic Colab download unavailable:', exc)
    print('Result ZIP remains at:', ZIP_PATH)
